# Module 4 — From Simulator to Hardware (runnable)

Signal scaling across domains, a canonical log record, and identical grading.

## Setup
Run this first. It defines the machine and the formulas using only Python's standard library, so the notebook runs anywhere with no `pip install`.

In [ ]:
# PKM curriculum — runnable Python reference (standard library only, no installs needed)
from math import pi, sqrt, hypot

# machine defaults (identical to the simulation engine)
B, L_CLOSED, STROKE = 0.6, 0.4, 0.6
BORE, ROD = 0.040, 0.022
SUPPLY, RELIEF = 16e6, 21e6
PUMP_MAX_FLOW, RATED_FLOW, RATED_DP = 6e-4, 2.5e-4, 3.5e6
PAYLOAD = 12.0

def inverse_kinematics(x, y, b=B): return hypot(x+b, y), hypot(x-b, y)
def forward_kinematics(L1, L2, b=B):
    x = (L1**2 - L2**2)/(4*b); return x, sqrt(max(0.0, L1**2 - (x+b)**2))
def det_jacobian(x, y, b=B):
    L1, L2 = inverse_kinematics(x, y, b); return 2*b*y/(L1*L2)
def manipulability(x, y, b=B): return abs(det_jacobian(x, y, b))
def cap_area(D=BORE): return pi*D**2/4
def rod_area(D=BORE, d=ROD): return pi*(D**2 - d**2)/4
def asymmetry(D=BORE, d=ROD): return cap_area(D)/rod_area(D, d)
def valve_flow(u, dP, Qr=RATED_FLOW, dPr=RATED_DP): return u*Qr*sqrt(max(0.0, dP/dPr))
print("reference loaded — stdlib only")

### Lesson 1.1/1.2 — Command and sensor scaling
Normalized command -> driver volts; sensor volts -> length.

In [ ]:
u = 0.7
print(f'command u={u} -> driver {10*u:+.1f} V')
k = 10/0.6                      # 0-10 V over 0-0.6 m
V = 7.5
print(f'sensor {V} V -> L = {L_CLOSED + V/k:.3f} m')

### Lesson 1.3 — Safety-chain thresholds
Simple guards that force a safe state, independent of the controller.

In [ ]:
def guards(p, L, manip):
    out=[]
    if p > RELIEF: out.append('OVER_PRESSURE')
    if not (L_CLOSED <= L <= L_CLOSED+STROKE): out.append('OVER_TRAVEL')
    if manip < 0.05: out.append('NEAR_SINGULAR')
    return out or ['ok']
print(guards(22e6, 0.9, 0.3))
print(guards(10e6, 0.9, 0.02))

### Lesson 2.1 — A canonical log record
One fixed schema both the simulator and hardware emit each cycle.

In [ ]:
t = 0.42
x, y = 0.098, 0.701
record = {'t': t, 'target': (0.10,0.70), 'pose': (x,y),
          'L': list(inverse_kinematics(x,y)), 'faults': []}
print(record)

### Lesson 2.2 — Grade a run from its log alone
The same rubric scores a sim log or a hardware log identically.

In [ ]:
def grade(overshoot, settling, max_os=0.15, max_ts=1.0):
    return 'PASS' if (overshoot <= max_os and settling <= max_ts) else 'FAIL'
print('sim  run:', grade(0.12, 0.80))
print('hw   run:', grade(0.13, 0.85))
print('ringing hw:', grade(0.20, 1.4))